In [3]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

from src.data_loading import load_years, harmonize, make_raw_archive

In [5]:
frames = load_years()

for y, df in frames.items():
    print(y, df.shape,
          "| salary labelled:", df["ConvertedCompYearly"].notna().sum(),
          "| Rwanda rows:", (df["Country"] == "Rwanda").sum())

print("Raw rows (all years):", sum(len(df) for df in frames.values()))

c:\Users\PeggyDusenge\Desktop\U\kodi-pay-summative\notebooks\..\src\data_loading.py:25: DtypeWarning: Columns (56: JobSatPoints_15_TEXT, 74: DatabaseHaveEntry, 92: DevEnvHaveEntry, 97: SOTagsHaveEntry, 98: SOTagsWant Entry, 105: OfficeStackWantEntry, 109: CommPlatformHaveEntr, 110: CommPlatformWantEntr, 132: SO_Actions_15_TEXT, 162: AIAgentOrchestration, 165: AIAgentObsWrite) have mixed types. Specify dtype option on import or set low_memory=False.
  return {y: pd.read_csv(download_data(y)) for y in years}


2023 (89184, 84) | salary labelled: 48019 | Rwanda rows: 27
2024 (65437, 114) | salary labelled: 23435 | Rwanda rows: 22
2025 (49191, 172) | salary labelled: 23947 | Rwanda rows: 15
Raw rows (all years): 203812


In [7]:
df = harmonize(frames)
print("Shared feature columns:", df.shape[1] - 1)
print(sorted(df.columns))

lab = df[df["ConvertedCompYearly"].notna()]
print("\nLabelled rows:", len(lab))
print("Rwanda labelled:", (lab["Country"] == "Rwanda").sum())

cc = lab.groupby("Country").size().sort_values(ascending=False)
for t in (50, 100, 200):
    print(f"Countries with >={t} labelled: {(cc >= t).sum()} | rows kept: {cc[cc >= t].sum()}")

print("\nTop 15 countries:\n", cc.head(15))
africa = ["Rwanda", "Kenya", "Nigeria", "Uganda", "Ghana", "South Africa", "Ethiopia", "Tanzania"]
print("\nAfrican countries:\n", cc[cc.index.isin(africa)])

Shared feature columns: 37
['AIAcc', 'AISelect', 'AISent', 'Age', 'CompTotal', 'ConvertedCompYearly', 'Country', 'Currency', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'DevType', 'EdLevel', 'Employment', 'ICorPM', 'Industry', 'LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'LearnCode', 'MainBranch', 'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith', 'OpSysPersonal use', 'OpSysProfessional use', 'OrgSize', 'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'PurchaseInfluence', 'RemoteWork', 'ResponseId', 'SOAccount', 'SOComm', 'SOPartFreq', 'SOVisitFreq', 'WebframeHaveWorkedWith', 'WebframeWantToWorkWith', 'WorkExp', 'Year', 'YearsCode']

Labelled rows: 95401
Rwanda labelled: 26
Countries with >=50 labelled: 87 | rows kept: 93985
Countries with >=100 labelled: 69 | rows kept: 92616
Countries with >=200 labelled: 57 | rows kept: 90791

Top 15 countries:
 Country
United States of America                                21801
Germany                           

In [9]:
lab = df[df["ConvertedCompYearly"].notna()].copy()

# 1. Who are the labelled respondents?
print(lab["MainBranch"].value_counts(), "\n")
print(lab["Employment"].value_counts().head(10), "\n")

# 2. Missing rate per feature per year (labelled rows only)
feature_cols = [c for c in df.columns if c not in
                ["ResponseId", "CompTotal", "Currency", "ConvertedCompYearly", "Year"]]
miss = lab.groupby("Year")[feature_cols].apply(lambda g: g.isna().mean()).T.round(2)
print(miss.sort_values(by=2025, ascending=False), "\n")

# 3. Salary distribution
print(lab["ConvertedCompYearly"].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))

MainBranch
I am a developer by profession                                                                86098
I am not primarily a developer, but I write code sometimes as part of my work/studies          7738
I used to be a developer by profession, but no longer am                                        498
I work with developers or my work supports developers but am not a developer by profession      390
I am learning to code                                                                           363
I code primarily as a hobby                                                                     314
Name: count, dtype: int64 

Employment
Employed, full-time                                                         56887
Employed                                                                    19500
Independent contractor, freelancer, or self-employed                         8580
Employed, full-time;Independent contractor, freelancer, or self-employed     4983
Employed, part-time   

In [10]:
import pandas as pd
from src.preprocessing import clean_and_label

data, funnel = clean_and_label(df)
print(funnel, "\n")

print("Countries kept:", data["Country"].nunique(), "\n")
print("Overall tier balance:\n", data["PayTier"].value_counts(normalize=True).round(3), "\n")
print("Tier balance by year:\n", pd.crosstab(data["Year"], data["PayTier"], normalize="index").round(3), "\n")

africa = ["Rwanda", "Kenya", "Nigeria", "Uganda", "Ghana", "South Africa", "Ethiopia"]
print("African countries kept:\n", data[data["Country"].isin(africa)].groupby("Country").size(), "\n")

for c in ["Age", "YearsCode", "EdLevel"]:
    print(data[c].value_counts().head(12), "\n")

                           Step    Rows
0               Raw pooled rows  203812
1       Respondents with salary   95401
2           Selected MainBranch   86098
3            Employed full-time   75357
4  Salary in [1,000, 1,000,000]   74014
5     Countries with >=100 rows   71612 

Countries kept: 63 

Overall tier balance:
 PayTier
High    0.334
Mid     0.333
Low     0.333
Name: proportion, dtype: float64 

Tier balance by year:
 PayTier    Low    Mid   High
Year                        
2023     0.342  0.342  0.316
2024     0.356  0.328  0.316
2025     0.285  0.319  0.395 

African countries kept:
 Country
Nigeria         142
South Africa    533
dtype: int64 

Age
25-34 years old       32834
35-44 years old       20821
18-24 years old        7656
45-54 years old        7316
55-64 years old        2575
65 years or older       344
Under 18 years old       33
Prefer not to say        33
Name: count, dtype: int64 

YearsCode
10    4823
15    3429
8     3414
20    2981
7     2974
12    2904